In [1]:
# Get raw data
with open('input/19.txt', 'r') as f:
    rawinput = f.read().strip()

In [2]:

# Part 1
scans = [[tuple([int(k) for k in j.split(',')])
          for j in i.split('\n')[1:]] 
         for i in rawinput.split('\n\n')]

pairs = dict()
for s in range(len(scans)):
    for i in range(len(scans[s])-1):
        for j in range(i+1,len(scans[s])):
            dist = sum([(k-l)**2 for k,l in zip(scans[s][i],scans[s][j])])
            if dist not in pairs:  pairs[dist] = {}
            pairs[dist][s] = (i,j)

get_overlaps = lambda x,y: [k for k in pairs 
                            if {*pairs[k]}.issuperset({x,y})]
overlaps = [[i,j]
            for i in range(len(scans)-1)
            for j in range(i+1,len(scans))
            if len(get_overlaps(i,j))>=12]

scanner_pos = {0: (0,0,0)}
beacons = {*scans[0]}
while len(scanner_pos) < len(scans):
    while True:
        if (z:=sum([k in scanner_pos for k in overlaps[0]]))==2:
            overlaps = overlaps[1:]
        elif z==0:
            overlaps = overlaps[1:]+overlaps[:1]
        else:
            iterpair = overlaps.pop(0)
            break
    s1,s2 = iterpair[::(iterpair[0] in scanner_pos)*2-1]

    pairmatch = [[pairs[p][s1],pairs[p][s2]] for p in get_overlaps(s1,s2)]
    ixconv = {}
    for i in {j for i in pairmatch for j in i[0]}:
        if len(m:=[(z:=z&l if k else l) 
                   for k,l in enumerate([{*j[1]} 
                                         for j in pairmatch 
                                         if i in j[0]])]) >= 3 and len(m[-1])==1:
            ixconv[i] = [*m[-1]][0]

    refpair = [i for i in pairmatch if all([j in ixconv for j in i[0]])][0][0]
    diffs1 = [j-k for j,k in zip(*[scans[s1][i] for i in refpair])]
    diffs2 = [j-k for j,k in zip(*[scans[s2][ixconv[i]] for i in refpair])]
    conv = [(j, i//diffs2[j]) for i in diffs1 for j in [0,1,2] if abs(diffs2[j])==abs(i)]
    
    scanner_pos[s2] = tuple([k-l 
                             for k,l in zip(scans[s1][refpair[0]],
                                            tuple([z[i]*j 
                                                   for z in [scans[s2][ixconv[refpair[0]]]] 
                                                   for i,j in conv]))])
    scans[s2] = [tuple([scanner_pos[s2][j]+i[conv[j][0]]*conv[j][1] 
                        for j in [0,1,2]]) 
                 for i in scans[s2]]
    beacons.update({*scans[s2]})
    
len(beacons)

353

In [3]:
# Part 2
max([sum([abs(l-m) for l,m in zip(*[scanner_pos[k] for k in [i,j]])])
     for i in range(len(scanner_pos)-1)
     for j in range(i+1,len(scanner_pos))])

10832